# CA05 - kNN Based Movie Recommender Engine
**Author:** Jerry Hsu  
**Course:** BSAN 6070 - Machine Learning  
**Institution:** Loyola Marymount University  
**Date:** Spring 2026

## Overview
This notebook builds a Movie Recommender System using the k-Nearest Neighbors (kNN) algorithm. Given a query movie, the system finds the 5 most similar movies based on genre features and IMDB ratings.

## Step 1: Import Libraries

In [12]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')
print("Libraries imported successfully!")

Libraries imported successfully!


## Step 2: Load the Data
We load the movie dataset directly from GitHub. The dataset contains 30 movies with 7 genre features and IMDB ratings.

> **Note:** If the GitHub URL is not working, you can use the local file `mmovies_recommendation_data.csv` included in this folder by replacing the url line with: `df = pd.read_csv("movies_recommendation_data.csv")`

In [14]:
url = "https://raw.githubusercontent.com/ArinB/MSBA-CA-Data/main/CA05/movies_recommendation_data.csv"
df = pd.read_csv(url)
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names: {list(df.columns)}")
df.head()

Dataset shape: (30, 11)

Column names: ['Movie ID', 'Movie Name', 'IMDB Rating', 'Biography', 'Drama', 'Thriller', 'Comedy', 'Crime', 'Mystery', 'History', 'Label']


,Movie ID,Movie Name,IMDB Rating,Biography,Drama,Thriller,Comedy,Crime,Mystery,History,Label
0,58,The Imitation Game,8.0,1,1,1,0,0,0,0,0
1,8,Ex Machina,7.7,0,1,0,0,0,1,0,0
2,46,A Beautiful Mind,8.2,1,1,0,0,0,0,0,0
3,62,Good Will Hunting,8.3,0,1,0,0,0,0,0,0
4,97,Forrest Gump,8.8,0,1,0,0,0,0,0,0


## Step 3: Explore the Data
We examine the dataset structure, check for missing values, and understand the feature distributions before building our recommender.

In [15]:
# Display basic info
print("=== Dataset Info ===")
print(df.info())
print("\n=== Missing Values ===")
print(df.isnull().sum())
print("\n=== Basic Statistics ===")
df.describe()

=== Dataset Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Movie ID     30 non-null     int64  
 1   Movie Name   30 non-null     object 
 2   IMDB Rating  30 non-null     float64
 3   Biography    30 non-null     int64  
 4   Drama        30 non-null     int64  
 5   Thriller     30 non-null     int64  
 6   Comedy       30 non-null     int64  
 7   Crime        30 non-null     int64  
 8   Mystery      30 non-null     int64  
 9   History      30 non-null     int64  
 10  Label        30 non-null     int64  
dtypes: float64(1), int64(9), object(1)
memory usage: 2.7+ KB
None

=== Missing Values ===
Movie ID       0
Movie Name     0
IMDB Rating    0
Biography      0
Drama          0
Thriller       0
Comedy         0
Crime          0
Mystery        0
History        0
Label          0
dtype: int64

=== Basic Statistics ===


,Movie ID,IMDB Rating,Biography,Drama,Thriller,Comedy,Crime,Mystery,History,Label
count,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.0
mean,48.133333,7.696667,0.233333,0.600000,0.100000,0.100000,0.133333,0.100000,0.100000,0.0
std,29.288969,0.666169,0.430183,0.498273,0.305129,0.305129,0.345746,0.305129,0.305129,0.0
min,1.000000,5.900000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
25%,27.750000,7.300000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
50%,48.500000,7.750000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
75%,64.250000,8.175000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
max,98.000000,8.800000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.0


## Step 4: Prepare the Feature Vector for "The Post"
We create the feature vector for the query movie "The Post" based on its known attributes:
- IMDB Rating: 7.2
- Biography: Yes (1)
- Drama: Yes (1)
- Thriller: No (0)
- Comedy: No (0)
- Crime: No (0)
- Mystery: No (0)
- History: Yes (1)

Note: The "History" column is not in the dataset features we train on — we only use the columns present in the dataset. We will align the query vector to match exactly the feature columns used by the model.

In [16]:
# Drop non-feature columns
feature_cols = ['IMDB Rating', 'Biography', 'Drama', 'Thriller', 'Comedy', 'Crime', 'Mystery']
X = df[feature_cols]

# Feature vector for "The Post"
the_post = pd.DataFrame([[7.2, 1, 1, 0, 0, 0, 0]], columns=feature_cols)

print("Feature columns used:", feature_cols)
print("\nFeature vector for 'The Post':")
print(the_post)

Feature columns used: ['IMDB Rating', 'Biography', 'Drama', 'Thriller', 'Comedy', 'Crime', 'Mystery']

Feature vector for 'The Post':
   IMDB Rating  Biography  Drama  Thriller  Comedy  Crime  Mystery
0          7.2          1      1         0       0      0        0


## Step 5: Build the kNN Model and Find Similar Movies
We use scikit-learn's NearestNeighbors to find the 5 most similar movies to "The Post" based on Euclidean distance across genre features and IMDB rating.

In [18]:
# Build kNN model
knn = NearestNeighbors(n_neighbors=6, metric='euclidean')
knn.fit(X)

# Find 5 nearest neighbors (6 because index 0 will be the query movie itself if it exists)
distances, indices = knn.kneighbors(the_post)

# Display recommendations
print("=== Top 5 Movie Recommendations for 'The Post' ===\n")
recommended_movies = df.iloc[indices[0]]['Movie Name'].values
recommended_distances = distances[0]

for i, (movie, dist) in enumerate(zip(recommended_movies, recommended_distances)):
    print(f"{i+1}. {movie}  (Distance: {dist:.4f})")

=== Top 5 Movie Recommendations for 'The Post' ===

1. Queen of Katwe  (Distance: 0.2000)
2. The Wind Rises  (Distance: 0.6000)
3. 12 Years a Slave  (Distance: 0.9000)
4. Hacksaw Ridge  (Distance: 1.0000)
5. A Beautiful Mind  (Distance: 1.0000)
6. The Karate Kid  (Distance: 1.0000)


## Results: Top 5 Movie Recommendations for "The Post"

Based on the kNN algorithm using Euclidean distance across 7 features (IMDB Rating, Biography, Drama, Thriller, Comedy, Crime, Mystery), the 5 most similar movies to **"The Post"** (IMDB: 7.2, Biography, Drama) are:

| Rank | Movie | Distance |
|------|-------|----------|
| 1 | Queen of Katwe | 0.2000 |
| 2 | The Wind Rises | 0.6000 |
| 3 | 12 Years a Slave | 0.9000 |
| 4 | Hacksaw Ridge | 1.0000 |
| 5 | A Beautiful Mind | 1.0000 |

### Interpretation
- **Queen of Katwe** is the closest match (distance: 0.2) — it shares Biography and Drama genres with a very similar IMDB rating
- **The Wind Rises** is the second closest (distance: 0.6) — sharing Biography and Drama genres
- **12 Years a Slave, Hacksaw Ridge, and A Beautiful Mind** all share at least one genre with "The Post" and have similar IMDB ratings

### Key Takeaway
The smaller the distance, the more similar the movie. Movies with matching genres AND similar IMDB ratings rank highest in our recommender system.